# 03 · Baseline (dummy classifiers)
Measures what a model that ignores the financial ratios would score, using 5-fold stratified cross-validation (CV) on the training set. Any real model in notebooks 04 and 05 must beat these numbers.

## 1. Load the training data
**What it does:** imports the libraries and loads `train.csv`, separating the 92 financial ratios (`X_train`) from the `Bankrupt?` target (`y_train`).

**Why:** only the training set is used here; the test set stays untouched until notebook 06.

In [5]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, confusion_matrix

pd.set_option("display.width", 200)

train = pd.read_csv("data/train.csv")
X_train = train.drop(columns="Bankrupt?")
y_train = train["Bankrupt?"]
print("Training data:", X_train.shape, "| bankrupt:", y_train.sum())

Training data: (5455, 92) | bankrupt: 176


## 2. Define the folds and the metrics
**What it does:** creates 5 stratified folds (shuffled, seed 42), each keeping the 3.2% bankruptcy rate, and lists the metrics to compute: accuracy, precision, recall, F1, ROC-AUC and PR-AUC (`average_precision`). `zero_division=0` reports 0 instead of an error when a model never predicts "bankrupt".

**Why:** 5-fold cross-validation (CV) scores every record with a model that never saw it, giving an honest estimate without using the test set. The same folds and metrics are reused in notebooks 04 and 05 so results are comparable.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy":  "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall":    "recall",
    "f1":        make_scorer(f1_score, zero_division=0),
    "roc_auc":   "roc_auc",
    "pr_auc":    "average_precision",
}

## 3. Cross-validate two dummy baselines
**What it does:** evaluates two models that ignore the financial ratios: one that always predicts "healthy" and one that guesses at random with the real 3.2% bankruptcy rate. It shows the average of each metric over the 5 folds.

**Why:** sets the minimum bar any real model must beat (ROC-AUC 0.50, PR-AUC about 0.03) and shows why accuracy is misleading: the "always healthy" model is 96.8% accurate but catches no bankruptcies.

In [7]:
models = {
    "Always 'healthy' (most_frequent)": DummyClassifier(strategy="most_frequent"),
    "Random guess (stratified)":        DummyClassifier(strategy="stratified", random_state=42),
}

results = {}
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    results[name] = {m: scores[f"test_{m}"].mean() for m in scoring}

pd.DataFrame(results).T.round(3)

,accuracy,precision,recall,f1,roc_auc,pr_auc
Always 'healthy' (most_frequent),0.968,0.000,0.000,0.000,0.500,0.032
Random guess (stratified),0.938,0.024,0.023,0.023,0.496,0.033


## 4. Confusion matrices
**What it does:** gives every training record one prediction from the fold where it was held out, then counts true and false predictions for each baseline. Rows are the actual outcome, columns the prediction (scikit-learn's layout).

**Why:** turns the percentages into counts of records: how many bankruptcies each baseline catches, misses and falsely flags.

In [8]:
for name, model in models.items():
    y_pred = cross_val_predict(model, X_train, y_train, cv=cv)
    cm = confusion_matrix(y_train, y_pred)
    print(f"\n{name}")
    print(pd.DataFrame(cm, index=["Actual: healthy", "Actual: bankrupt"],
                           columns=["Predicted: healthy", "Predicted: bankrupt"]))


Always 'healthy' (most_frequent)
                  Predicted: healthy  Predicted: bankrupt
Actual: healthy                 5279                    0
Actual: bankrupt                 176                    0

Random guess (stratified)
                  Predicted: healthy  Predicted: bankrupt
Actual: healthy                 5113                  166
Actual: bankrupt                 172                    4
